# VLM Validation Pipeline (Colab)

Этот ноутбук:
- запускает единый пайплайн вызова VLM-моделей;
- валидирует извлечение физических свойств объекта;
- считает для каждого свойства, где модель дала значение (`yes`) и где не дала (`no`);
- считает проценты покрытия: что модель выделяет, а что упускает.

Основной источник GT: `dataset/meta.json`.


In [ ]:
# Dependencies (Colab-safe versions + automatic runtime restart once)
import os
import sys
import subprocess
from pathlib import Path

MARKER = Path('/tmp/vlm_coursework_deps_ready')

if not MARKER.exists():
    print('Installing dependencies (first run)...')

    install_cmds = [
        [sys.executable, '-m', 'pip', 'uninstall', '-y', 'numpy'],
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall', 'numpy==2.1.3'],
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade', '--upgrade-strategy', 'only-if-needed',
         'transformers>=4.49.0,<5.0.0', 'accelerate', 'bitsandbytes', 'sentencepiece'],
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade', '--upgrade-strategy', 'only-if-needed',
         'pandas==2.2.2', 'pillow<12', 'boto3', 'rembg', 'onnxruntime', 'comet_ml'],
    ]

    for cmd in install_cmds:
        print('>>', ' '.join(cmd))
        subprocess.run(cmd, check=True)

    MARKER.write_text('ok')
    print('Dependencies installed. Restarting runtime now...')

    # Colab-safe hard restart so new binary wheels are loaded cleanly
    os.kill(os.getpid(), 9)
else:
    print('Dependencies already installed in this runtime. Continue.')


In [ ]:
# Clone or update public repo in Colab
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/Yaitco/VLM-2D-Physics-Boundaries.git"
WORKDIR = Path("/content/VLM-2D-Physics-Boundaries")

if not WORKDIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
else:
    print(f"Repo already exists: {WORKDIR}. Pulling latest...")
    subprocess.run(["git", "-C", str(WORKDIR), "pull", "--ff-only"], check=True)

%cd /content/VLM-2D-Physics-Boundaries


In [ ]:
# Optional: regenerate ABO subset for physics validation (+ masks)
import subprocess

REGENERATE_DATASET = False
DOWNLOAD_MISSING = True
GENERATE_MASKS = True
MASK_BACKEND = "rembg"  # rembg | simple
EXCLUDE_PRODUCT_TYPES = "CELLULAR_PHONE_CASE,PORTABLE_ELECTRONIC_DEVICE_COVER"

if REGENERATE_DATASET:
    cmd = [
        "python",
        "scripts/build_abo_physics_subset.py",
        "--dataset-root", "dataset",
        "--cache-dir", "dataset/abo_vlm_val/_cache",
        "--source-images-dir", "dataset/abo_vlm_val/images",
        "--listing-shards", "0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15",
        "--max-samples", "240",
        "--min-known-properties", "4",
        "--max-per-product-type", "25",
        "--exclude-product-types", EXCLUDE_PRODUCT_TYPES,
        "--seed", "42",
        "--clear-output",
    ]
    if DOWNLOAD_MISSING:
        cmd.append("--download-missing")
    if GENERATE_MASKS:
        cmd += [
            "--generate-masks",
            "--mask-backend", MASK_BACKEND,
            "--min-mask-area-ratio", "0.01",
            "--max-mask-area-ratio", "0.95",
        ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Skip regeneration. Set REGENERATE_DATASET=True to rebuild dataset.")


In [ ]:
import gc
import os
import json
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

from transformers import (
    AutoModelForImageTextToText,
    AutoModelForCausalLM,
    AutoProcessor,
    BitsAndBytesConfig,
)

# ---------- Task schema ----------
PROPERTY_VALUES = {
    "material": {"glass", "metal", "wood", "plastic", "rubber", "fabric", "paper", "ceramic", "stone", "mixed", "unknown"},
    "rigidity": {"rigid", "soft", "flexible", "mixed", "unknown"},
    "transparency": {"opaque", "transparent", "translucent", "unknown"},
    "surface": {"smooth", "rough", "fuzzy", "porous", "mixed", "unknown"},
    "fragility": {"fragile", "durable", "unknown"},
}
PROPERTY_ORDER = ["material", "rigidity", "transparency", "surface", "fragility"]

# ---------- Data ----------
ROOT = Path("dataset")
DATASET_NAME = "abo_physics_val"
META_PATH = ROOT / DATASET_NAME / "meta.json"
FALLBACK_META_PATH = ROOT / "meta.json"
REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if not META_PATH.exists():
    if FALLBACK_META_PATH.exists():
        print(f"Warning: {META_PATH} not found. Using fallback {FALLBACK_META_PATH}")
        META_PATH = FALLBACK_META_PATH
    else:
        raise FileNotFoundError(
            f"Dataset meta not found: {META_PATH}. Run scripts/build_abo_physics_subset.py first."
        )

# ---------- Segmentation experiment config ----------
# raw: original image with background
# masked: apply segmentation mask and keep only primary object
EVAL_VARIANTS = ["raw", "masked"]
MASK_BACKGROUND_MODE = "black"  # black | white

# ---------- Model registry (single unified pipeline) ----------
MODEL_REGISTRY = {
    "qwen3_vl_8b": {
        "backend": "hf_chat",
        "model_id": "Qwen/Qwen3-VL-8B-Instruct",
        "use_4bit": True,
        "max_new_tokens": 256,
    },
    "qwen2_5_vl_7b": {
        "backend": "hf_chat",
        "model_id": "Qwen/Qwen2.5-VL-7B-Instruct",
        "use_4bit": True,
        "max_new_tokens": 256,
    },
    "llava_onevision_1_5_8b": {
        "backend": "hf_chat",
        "model_id": "lmms-lab/LLaVA-OneVision-1.5-8B-Instruct",
        "use_4bit": True,
        "max_new_tokens": 256,
    },
}

# Pick model here
SELECTED_MODEL = "qwen3_vl_8b"

# Evaluation config
MAX_SAMPLES: Optional[int] = None
RANDOM_SEED = 42
SAVE_RAW_OUTPUT = True

assert SELECTED_MODEL in MODEL_REGISTRY, f"Unknown model key: {SELECTED_MODEL}"


# ---------- Comet logging ----------
COMET_ENABLED = True
COMET_DEFAULT_PROJECT = "vlm-physics-validation"


def get_secret_value(secret_name: str) -> Optional[str]:
    try:
        from google.colab import userdata

        value = userdata.get(secret_name)
        if value:
            return value
    except Exception:
        pass

    return os.getenv(secret_name.upper()) or os.getenv(secret_name)


def init_comet_experiment(run_tag: str):
    if not COMET_ENABLED:
        print("Comet: disabled by COMET_ENABLED=False")
        return None

    api_key = get_secret_value("comet_api_key")
    workspace = get_secret_value("comet_workspace")
    project_name = get_secret_value("comet_project_name") or COMET_DEFAULT_PROJECT

    if not api_key:
        print(
            "Comet: API key is missing. "
            "Add 'comet_api_key' in Colab userdata (Secrets) or set COMET_API_KEY env var."
        )
        return None

    try:
        from comet_ml import Experiment
    except Exception as exc:
        print(f"Comet: comet_ml import failed: {exc}")
        return None

    kwargs = {
        "api_key": api_key,
        "project_name": project_name,
        "auto_output_logging": "simple",
    }
    if workspace:
        kwargs["workspace"] = workspace

    try:
        exp = Experiment(**kwargs)
    except Exception as exc:
        print(f"Comet: failed to initialize Experiment: {exc}")
        return None

    exp.set_name(run_tag)
    exp.log_parameters(
        {
            "dataset_name": DATASET_NAME,
            "meta_path": str(META_PATH),
            "selected_model_key": SELECTED_MODEL,
            "selected_model_id": MODEL_REGISTRY[SELECTED_MODEL]["model_id"],
            "eval_variants_requested": ",".join(EVAL_VARIANTS),
            "mask_background_mode": MASK_BACKGROUND_MODE,
            "max_samples": MAX_SAMPLES,
            "random_seed": RANDOM_SEED,
        }
    )

    print(f"Comet: logging to project='{project_name}' workspace='{workspace}'")
    return exp


In [ ]:
with open(META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

if MAX_SAMPLES is not None and len(meta) > MAX_SAMPLES:
    rng = random.Random(RANDOM_SEED)
    meta = rng.sample(meta, k=MAX_SAMPLES)

print(f"Samples for validation: {len(meta)}")
print(json.dumps(meta[0], ensure_ascii=False, indent=2)[:1000])

prompt_template = """
You are given an image.

The image_id is "{image_id}". Output must contain exactly this image_id.

Step 1: Choose the PRIMARY OBJECT (the main item in the foreground).
Ignore background items.

Step 2: Fill properties for the PRIMARY OBJECT ONLY.

Definitions:
- material = the object's substance (what it is made of),
  NOT what it is covered by and NOT the background.
- surface = visible texture or finish of the object's outer appearance
  (e.g., fuzzy, rough).
- If the object is a living being (animal or person),
  set material = "mixed".

Return ONLY valid JSON with EXACTLY this structure:

{{
  "image_id": "{image_id}",
  "primary_object": "<short noun phrase>",
  "properties": {{
    "material": "<one of: glass|metal|wood|plastic|rubber|fabric|paper|ceramic|stone|mixed|unknown>",
    "rigidity": "<one of: rigid|soft|flexible|mixed|unknown>",
    "transparency": "<one of: opaque|transparent|translucent|unknown>",
    "surface": "<one of: smooth|rough|fuzzy|porous|mixed|unknown>",
    "fragility": "<one of: fragile|durable|unknown>"
  }},
  "notes": "<3-10 words: visual cues for the object only>"
}}

Rules:
- Use ONLY the listed values.
- If uncertain from the image alone, use "unknown".
- Do NOT mention background objects or materials in notes.
- Do NOT use world knowledge or object function.
- No extra text. JSON only.
""".strip()


In [ ]:
@dataclass
class VLMRuntime:
    name: str
    backend: str
    model_id: str
    processor: Any
    model: Any
    gen_kwargs: Dict[str, Any]


def make_bnb_config(use_4bit: bool):
    if not use_4bit:
        return None
    if not torch.cuda.is_available():
        print("CUDA is not available. 4-bit quantization disabled.")
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )


def load_hf_chat_runtime(name: str, cfg: Dict[str, Any]) -> VLMRuntime:
    model_id = cfg["model_id"]
    use_4bit = bool(cfg.get("use_4bit", True))
    gen_kwargs = {
        "max_new_tokens": int(cfg.get("max_new_tokens", 256)),
        "do_sample": bool(cfg.get("do_sample", False)),
    }

    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

    model_kwargs = {
        "device_map": "auto",
        "trust_remote_code": True,
    }

    bnb = make_bnb_config(use_4bit)
    if bnb is not None:
        model_kwargs["quantization_config"] = bnb
    elif torch.cuda.is_available():
        model_kwargs["torch_dtype"] = torch.float16

    try:
        model = AutoModelForImageTextToText.from_pretrained(model_id, **model_kwargs)
    except Exception as e:
        print(f"AutoModelForImageTextToText failed: {e}")
        print("Falling back to AutoModelForCausalLM...")
        model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)

    model.eval()
    return VLMRuntime(
        name=name,
        backend="hf_chat",
        model_id=model_id,
        processor=processor,
        model=model,
        gen_kwargs=gen_kwargs,
    )


def infer_hf_chat(runtime: VLMRuntime, image: Image.Image, prompt: str) -> str:
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    if hasattr(runtime.processor, "apply_chat_template"):
        text = runtime.processor.apply_chat_template(messages, add_generation_prompt=True)
    else:
        text = prompt

    inputs = runtime.processor(
        text=text,
        images=image,
        return_tensors="pt",
        truncation=False,
    )

    if hasattr(runtime.model, "device") and str(runtime.model.device) != "meta":
        inputs = {k: v.to(runtime.model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = runtime.model.generate(**inputs, **runtime.gen_kwargs)

    input_len = inputs["input_ids"].shape[1]
    gen = out[:, input_len:]
    return runtime.processor.batch_decode(gen, skip_special_tokens=True)[0]


BACKEND_LOADERS = {
    "hf_chat": load_hf_chat_runtime,
}

BACKEND_INFER = {
    "hf_chat": infer_hf_chat,
}


def load_runtime(model_key: str) -> VLMRuntime:
    cfg = MODEL_REGISTRY[model_key]
    backend = cfg["backend"]
    loader = BACKEND_LOADERS[backend]
    runtime = loader(model_key, cfg)
    print(f"Loaded model: {runtime.name} -> {runtime.model_id}")
    return runtime


def infer_runtime(runtime: VLMRuntime, image: Image.Image, prompt: str) -> str:
    return BACKEND_INFER[runtime.backend](runtime, image, prompt)


def unload_runtime(runtime: Optional[VLMRuntime]):
    if runtime is None:
        return
    try:
        del runtime.model
    except Exception:
        pass
    try:
        del runtime.processor
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
def strip_code_fences(text: str) -> str:
    t = text.strip()
    if t.startswith("```"):
        t = re.sub(r"^```(?:json)?\s*", "", t, flags=re.IGNORECASE)
        t = re.sub(r"\s*```$", "", t)
    return t.strip()


def find_first_json_object(text: str) -> Optional[str]:
    s = strip_code_fences(text)
    start = s.find("{")
    if start < 0:
        return None

    depth = 0
    in_str = False
    escape = False

    for i in range(start, len(s)):
        ch = s[i]
        if in_str:
            if escape:
                escape = False
            elif ch == "\\":
                escape = True
            elif ch == '"':
                in_str = False
            continue

        if ch == '"':
            in_str = True
        elif ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return s[start:i + 1]

    return None


def parse_model_output(raw_text: str):
    blob = find_first_json_object(raw_text)
    if blob is None:
        return None, "json_not_found"
    try:
        return json.loads(blob), None
    except Exception as e:
        return None, f"json_parse_error: {e}"


def norm_value(prop: str, value: Any) -> str:
    if value is None:
        return "unknown"
    v = str(value).strip().lower()
    if not v:
        return "unknown"
    if v in PROPERTY_VALUES[prop]:
        return v
    return "unknown"


def normalize_gt_properties(sample_meta: Dict[str, Any]) -> Dict[str, str]:
    props = sample_meta.get("properties", {})
    if not isinstance(props, dict):
        props = {}
    return {p: norm_value(p, props.get(p)) for p in PROPERTY_ORDER}


def normalize_pred(parsed_json: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    if not isinstance(parsed_json, dict):
        return {
            "image_id": None,
            "primary_object": None,
            "notes": None,
            "properties": {p: "unknown" for p in PROPERTY_ORDER},
        }

    props = parsed_json.get("properties", {})
    if not isinstance(props, dict):
        props = {}

    return {
        "image_id": str(parsed_json.get("image_id")).strip() if parsed_json.get("image_id") is not None else None,
        "primary_object": parsed_json.get("primary_object"),
        "notes": parsed_json.get("notes"),
        "properties": {p: norm_value(p, props.get(p)) for p in PROPERTY_ORDER},
    }


In [ ]:
def apply_mask_to_image(image: Image.Image, mask: Image.Image, bg_mode: str = "black") -> Image.Image:
    rgb = np.asarray(image.convert("RGB"), dtype=np.uint8)
    m = np.asarray(mask.convert("L"), dtype=np.uint8) > 127

    out = rgb.copy()
    if bg_mode == "white":
        out[~m] = 255
    else:
        out[~m] = 0

    return Image.fromarray(out, mode="RGB")


def load_variant_image(root: Path, sample_meta: Dict[str, Any], variant: str) -> Image.Image:
    image_path = root / sample_meta["path"]
    image = Image.open(image_path).convert("RGB")

    if variant == "raw":
        return image

    if variant == "masked":
        mask_rel = sample_meta.get("mask_path")
        if not mask_rel:
            raise FileNotFoundError("mask_path is missing in sample metadata")
        mask_path = root / mask_rel
        if not mask_path.exists():
            raise FileNotFoundError(f"Mask file not found: {mask_path}")
        mask = Image.open(mask_path).convert("L")
        return apply_mask_to_image(image, mask, bg_mode=MASK_BACKGROUND_MODE)

    raise ValueError(f"Unknown variant: {variant}")


def evaluate_one(runtime: VLMRuntime, root: Path, sample_meta: Dict[str, Any], variant: str) -> Dict[str, Any]:
    image_id = str(sample_meta["image_id"])
    image = load_variant_image(root, sample_meta, variant)

    prompt = prompt_template.format(image_id=image_id)
    raw = infer_runtime(runtime, image, prompt)

    parsed_json, parse_error = parse_model_output(raw)
    pred = normalize_pred(parsed_json)
    gt_props = normalize_gt_properties(sample_meta)
    pred_props = pred["properties"]

    row = {
        "variant": variant,
        "image_id": image_id,
        "path": sample_meta.get("path"),
        "mask_path": sample_meta.get("mask_path"),
        "has_valid_json": parse_error is None,
        "parse_error": parse_error,
        "image_id_matched": pred.get("image_id") == image_id,
        "primary_object_pred": pred.get("primary_object"),
        "notes_pred": pred.get("notes"),
    }

    for prop in PROPERTY_ORDER:
        gt = gt_props[prop]
        pv = pred_props[prop]
        gt_known = gt != "unknown"
        pred_known = pv != "unknown"

        row[f"{prop}_gt"] = gt
        row[f"{prop}_pred"] = pv
        row[f"{prop}_gt_known"] = gt_known
        row[f"{prop}_pred_known"] = pred_known
        row[f"{prop}_missed_when_gt_known"] = gt_known and (not pred_known)
        row[f"{prop}_exact_match"] = gt_known and (pv == gt)

    if SAVE_RAW_OUTPUT:
        row["raw_output"] = raw

    return row


def run_validation(model_key: str, samples: List[Dict[str, Any]], variant: str = "raw") -> pd.DataFrame:
    runtime = None
    rows = []

    try:
        runtime = load_runtime(model_key)

        for sample_meta in tqdm(samples, desc=f"Evaluating {model_key} [{variant}]"):
            try:
                row = evaluate_one(runtime, ROOT, sample_meta, variant=variant)
            except Exception as e:
                image_id = str(sample_meta.get("image_id", "unknown"))
                row = {
                    "variant": variant,
                    "image_id": image_id,
                    "path": sample_meta.get("path"),
                    "mask_path": sample_meta.get("mask_path"),
                    "has_valid_json": False,
                    "parse_error": f"runtime_error: {e}",
                    "image_id_matched": False,
                    "primary_object_pred": None,
                    "notes_pred": None,
                }
                gt_props = normalize_gt_properties(sample_meta)
                for prop in PROPERTY_ORDER:
                    gt = gt_props[prop]
                    gt_known = gt != "unknown"
                    row[f"{prop}_gt"] = gt
                    row[f"{prop}_pred"] = "unknown"
                    row[f"{prop}_gt_known"] = gt_known
                    row[f"{prop}_pred_known"] = False
                    row[f"{prop}_missed_when_gt_known"] = gt_known
                    row[f"{prop}_exact_match"] = False
                if SAVE_RAW_OUTPUT:
                    row["raw_output"] = None

            rows.append(row)

    finally:
        unload_runtime(runtime)

    return pd.DataFrame(rows)


In [ ]:
def build_property_metrics(df: pd.DataFrame) -> pd.DataFrame:
    total = len(df)
    metrics = []

    for prop in PROPERTY_ORDER:
        pred_yes = int(df[f"{prop}_pred_known"].sum())
        pred_no = int(total - pred_yes)

        gt_known = int(df[f"{prop}_gt_known"].sum())
        missed = int(df[f"{prop}_missed_when_gt_known"].sum())
        extracted_when_gt_known = int(gt_known - missed)

        exact = int(df[f"{prop}_exact_match"].sum())

        metrics.append(
            {
                "property": prop,
                "pred_yes_count": pred_yes,
                "pred_no_count": pred_no,
                "pred_yes_pct": round(100.0 * pred_yes / total, 2) if total else 0.0,
                "gt_known_count": gt_known,
                "extracted_when_gt_known": extracted_when_gt_known,
                "missed_when_gt_known": missed,
                "coverage_on_gt_known_pct": round(100.0 * extracted_when_gt_known / gt_known, 2) if gt_known else None,
                "exact_match_on_gt_known_pct": round(100.0 * exact / gt_known, 2) if gt_known else None,
            }
        )

    return pd.DataFrame(metrics)


def log_report_to_comet(
    comet_experiment,
    model_key: str,
    variant: str,
    property_metrics: pd.DataFrame,
    summary: Dict[str, Any],
    per_sample_path: Path,
    property_path: Path,
    summary_path: Path,
):
    if comet_experiment is None:
        return

    summary_metrics = {
        f"{model_key}/{variant}/num_samples": summary["num_samples"],
        f"{model_key}/{variant}/valid_json_pct": summary["valid_json_pct"],
        f"{model_key}/{variant}/image_id_match_pct": summary["image_id_match_pct"],
    }
    comet_experiment.log_metrics(summary_metrics)

    for _, row in property_metrics.iterrows():
        prop = row["property"]
        for col in [
            "pred_yes_pct",
            "coverage_on_gt_known_pct",
            "exact_match_on_gt_known_pct",
            "pred_yes_count",
            "pred_no_count",
            "missed_when_gt_known",
        ]:
            val = row[col]
            if pd.isna(val):
                continue
            comet_experiment.log_metric(f"{model_key}/{variant}/{prop}/{col}", float(val))

    # Keep artifacts for later audit.
    comet_experiment.log_asset(str(per_sample_path), file_name=f"{model_key}_{variant}_per_sample_predictions.csv")
    comet_experiment.log_asset(str(property_path), file_name=f"{model_key}_{variant}_property_metrics.csv")
    comet_experiment.log_asset(str(summary_path), file_name=f"{model_key}_{variant}_summary.json")


def save_report(df: pd.DataFrame, model_key: str, variant: str, comet_experiment=None):
    run_dir = REPORTS_DIR / model_key / variant
    run_dir.mkdir(parents=True, exist_ok=True)

    property_metrics = build_property_metrics(df)

    per_sample_path = run_dir / "per_sample_predictions.csv"
    property_path = run_dir / "property_metrics.csv"
    summary_path = run_dir / "summary.json"

    df.to_csv(per_sample_path, index=False)
    property_metrics.to_csv(property_path, index=False)

    summary = {
        "model_key": model_key,
        "model_id": MODEL_REGISTRY[model_key]["model_id"],
        "variant": variant,
        "num_samples": int(len(df)),
        "valid_json_count": int(df["has_valid_json"].sum()),
        "valid_json_pct": round(100.0 * float(df["has_valid_json"].mean()), 2) if len(df) else 0.0,
        "image_id_match_count": int(df["image_id_matched"].sum()),
        "image_id_match_pct": round(100.0 * float(df["image_id_matched"].mean()), 2) if len(df) else 0.0,
    }

    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"\n=== Property coverage ({variant}) ===")
    display(property_metrics)

    print("\n=== Run summary ===")
    print(json.dumps(summary, ensure_ascii=False, indent=2))

    print("\nSaved:")
    print(f"- {per_sample_path}")
    print(f"- {property_path}")
    print(f"- {summary_path}")

    log_report_to_comet(
        comet_experiment=comet_experiment,
        model_key=model_key,
        variant=variant,
        property_metrics=property_metrics,
        summary=summary,
        per_sample_path=per_sample_path,
        property_path=property_path,
        summary_path=summary_path,
    )

    return property_metrics, summary


available_variants = []
if "raw" in EVAL_VARIANTS:
    available_variants.append("raw")
if "masked" in EVAL_VARIANTS and any("mask_path" in x for x in meta):
    available_variants.append("masked")

print("Running variants:", available_variants)

comet_experiment = init_comet_experiment(run_tag=f"{SELECTED_MODEL}_{DATASET_NAME}")

try:
    variant_metrics = []
    variant_summaries = []
    for variant in available_variants:
        df_variant = run_validation(SELECTED_MODEL, meta, variant=variant)
        pm_variant, summary_variant = save_report(
            df_variant,
            SELECTED_MODEL,
            variant,
            comet_experiment=comet_experiment,
        )
        pm_variant = pm_variant.copy()
        pm_variant["variant"] = variant
        variant_metrics.append(pm_variant)
        variant_summaries.append(summary_variant)

    if variant_metrics:
        all_variant_metrics_df = pd.concat(variant_metrics, ignore_index=True)
        display(all_variant_metrics_df)

        if set(available_variants) >= {"raw", "masked"}:
            pivot = all_variant_metrics_df.pivot(index="property", columns="variant", values="coverage_on_gt_known_pct")
            if "raw" in pivot.columns and "masked" in pivot.columns:
                pivot["delta_masked_minus_raw"] = pivot["masked"] - pivot["raw"]
            print("\nCoverage comparison (masked vs raw):")
            display(pivot)

            if comet_experiment is not None and "delta_masked_minus_raw" in pivot.columns:
                for prop, row in pivot.iterrows():
                    delta = row["delta_masked_minus_raw"]
                    if pd.notna(delta):
                        comet_experiment.log_metric(
                            f"{SELECTED_MODEL}/comparison/{prop}/delta_masked_minus_raw",
                            float(delta),
                        )
finally:
    if comet_experiment is not None:
        comet_experiment.end()


In [ ]:
# Optional: compare several models and both variants

def run_many_models(
    model_keys: List[str],
    samples: List[Dict[str, Any]],
    variants: Optional[List[str]] = None,
    log_to_comet: bool = False,
) -> pd.DataFrame:
    if variants is None:
        variants = ["raw"]
        if any("mask_path" in x for x in samples):
            variants.append("masked")

    comet_experiment = None
    if log_to_comet:
        comet_experiment = init_comet_experiment(run_tag="multi_model_comparison")

    rows = []
    try:
        for mk in model_keys:
            for variant in variants:
                print(f"\n######## Running {mk} [{variant}] ########")
                df = run_validation(mk, samples, variant=variant)
                pm, summary = save_report(df, mk, variant, comet_experiment=comet_experiment)
                for _, r in pm.iterrows():
                    rows.append(
                        {
                            "model_key": mk,
                            "variant": variant,
                            "property": r["property"],
                            "pred_yes_pct": r["pred_yes_pct"],
                            "coverage_on_gt_known_pct": r["coverage_on_gt_known_pct"],
                            "exact_match_on_gt_known_pct": r["exact_match_on_gt_known_pct"],
                            "num_samples": summary["num_samples"],
                        }
                    )
    finally:
        if comet_experiment is not None:
            comet_experiment.end()

    return pd.DataFrame(rows)


# Example usage:
# comparison_df = run_many_models(
#     ["qwen3_vl_8b", "qwen2_5_vl_7b", "llava_onevision_1_5_8b"],
#     meta,
#     variants=["raw", "masked"],
#     log_to_comet=True,
# )
# display(comparison_df)


## Как расширять на другие модели

1. Добавьте модель в `MODEL_REGISTRY`.
2. Если модель совместима с chat/image интерфейсом HF, достаточно `backend="hf_chat"`.
3. Если нужна другая логика вызова, добавьте новый backend в `BACKEND_LOADERS` и `BACKEND_INFER`.

Таким образом один и тот же evaluation-код и метрики переиспользуются для всех моделей.
